# **Musae** — MUon Scattering and Absorption tomography simulation infrastructurE
# Copyright (C) 2026 Musae developers
#
# ML-EM Reconstruction Visualization

In [36]:
import numpy as np
import pandas as pd
import json
import os
from matplotlib import pyplot as plt
from matplotlib.colors import LogNorm
from matplotlib.ticker import ScalarFormatter
import plotly.graph_objs as go
import plotly.io as pio
import math
import numba
%matplotlib ipympl

In [37]:
# ============================================================
# Load Reconstruction Result
# ============================================================

# --- Set the .npy file path ---
file_path = "output/ML_EM/Scatter_cube_5p4e9_median_discard_N50_06-22.npy"
# file_path = "output/ML_EM_PData/ML_EM_mean_discard_N50_06-22.npy"

base_name = os.path.splitext(file_path)[0]
txt_file = base_name + ".txt"
rho = np.load(file_path)
# rho = np.load('test_data_ground_truth.npy')

with open(txt_file, 'r') as f:
    params = json.load(f)

# --- Extract parameters ---
X_MIN = params["X_MIN"]
X_MAX = params["X_MAX"]
Y_MIN = params["Y_MIN"]
Y_MAX = params["Y_MAX"]
Z_MIN = params["Z_MIN"]
Z_MAX = params["Z_MAX"]
N_X = params["N_X"]
N_Y = params["N_Y"]
N_Z = params["N_Z"]
Method = params.get("Method", "ML-EM")
N_iter = params.get("N_iter", "?")
Lambda_init = params.get("Lambda_init", "?")
PoCA_handling = params.get("PoCA_handling", "N/A")
Lambda_final_mean = params.get("Lambda_final_mean", 
    params.get("Lambda_final_mean_seen", "?"))  # support both old/new param names
Lambda_final_max = params.get("Lambda_final_max", "?")
N_events_total = params.get("N_events_total", "?")
N_events_valid = params.get("N_events_valid_original", "?")
N_events_effective = params.get("N_events_effective", "?")
N_events_discarded = params.get("N_events_discarded", "?")
N_voxels_seen = params.get("N_voxels_seen", "?")
N_voxels_unseen = params.get("N_voxels_unseen", "?")
Sentinel = params.get("Sentinel_unseen", -1.0)
P0_MeV = params.get("P0_MeV", "?")
Unit = params.get("Unit", "(cm), (mrad^2/cm)")

# Mask: seen voxels (λ >= 0) vs unseen sentinel (λ == -1)
seen_mask = rho >= 0
n_seen = np.sum(seen_mask)

print(f"=== ML-EM Reconstruction Result ===")
print(f"Method: {Method}, Iterations: {N_iter}")
print(f"PoCA handling: {PoCA_handling}")
print(f"Grid: {N_X} × {N_Y} × {N_Z} voxels")
print(f"Volume: X[{X_MIN}, {X_MAX}] × Y[{Y_MIN}, {Y_MAX}] × Z[{Z_MIN}, {Z_MAX}] {Unit.split(',')[0]}")
print(f"Events: {N_events_valid}/{N_events_total} valid")
print(f"Effective events used: {N_events_effective}, Discarded: {N_events_discarded}")
print(f"P0 = {P0_MeV} MeV/c, λ_init = {Lambda_init} mrad²/cm")
print(f"Seen voxels: {n_seen}/{N_X*N_Y*N_Z}, Unseen (sentinel={Sentinel}): {N_X*N_Y*N_Z - n_seen}")
print(f"Result λ (seen only): mean={rho[seen_mask].mean():.4f}, max={rho[seen_mask].max():.4f} mrad²/cm")
print(f"rho shape: {rho.shape}")
print(f"rho range (all): [{rho.min():.4f}, {rho.max():.4f}]")

=== ML-EM Reconstruction Result ===
Method: ML-EM-median, Iterations: 50
PoCA handling: discard
Grid: 10 × 10 × 10 voxels
Volume: X[-30.0, 30.0] × Y[-30.0, 30.0] × Z[-30.0, 30.0] (cm)
Events: 7740482/8097938 valid
Effective events used: 3482059, Discarded: 4258423
P0 = 1000.0 MeV/c, λ_init = 1.0 mrad²/cm
Seen voxels: 1000/1000, Unseen (sentinel=-1.0): 0
Result λ (seen only): mean=0.7796, max=20.9644 mrad²/cm
rho shape: (10, 10, 10)
rho range (all): [0.1035, 20.9644]


In [38]:
# ============================================================
# Build 3D Grid
# ============================================================

dx = (X_MAX - X_MIN) / N_X
dy = (Y_MAX - Y_MIN) / N_Y
dz = (Z_MAX - Z_MIN) / N_Z

x = np.linspace(X_MIN + dx/2, X_MAX - dx/2, N_X, dtype=float)
y = np.linspace(Y_MIN + dy/2, Y_MAX - dy/2, N_Y, dtype=float)
z = np.linspace(Z_MIN + dz/2, Z_MAX - dz/2, N_Z, dtype=float)

X_CUBE, Y_CUBE, Z_CUBE = np.meshgrid(x, y, z, indexing='ij')

print(f"Voxel size: dx={dx:.2f}, dy={dy:.2f}, dz={dz:.2f} {Unit.split(',')[0]}")
print(f"Grid shapes: X={X_CUBE.shape}, rho={rho.shape}")

Voxel size: dx=6.00, dy=6.00, dz=6.00 (cm)
Grid shapes: X=(10, 10, 10), rho=(10, 10, 10)


In [39]:
def plot_cube_edges(vertices, color='red', width=2):
    """基于立方体 8 个顶点绘制 12 条棱线 (Plotly 3D 线框)"""
    edges = [
        [0, 1], [1, 2], [2, 3], [3, 0],  # 底面
        [4, 5], [5, 6], [6, 7], [7, 4],  # 顶面
        [0, 4], [1, 5], [2, 6], [3, 7]   # 侧棱
    ]
    traces = []
    for v1_idx, v2_idx in edges:
        v1, v2 = vertices[v1_idx], vertices[v2_idx]
        traces.append(go.Scatter3d(
            x=[v1[0], v2[0]], y=[v1[1], v2[1]], z=[v1[2], v2[2]],
            mode='lines', line=dict(color=color, width=width),
            showlegend=False
        ))
    return traces


def plot_cube_edges_legend(size, color, name):
    """为线框立方体创建图例项"""
    return go.Scatter3d(
        x=[None], y=[None], z=[None],
        mode='markers',
        marker=dict(symbol='square-open', size=size, color=color, line=dict(width=3)),
        name=name, showlegend=True,
    )


def PrintMLResult(X_CUBE, Y_CUBE, Z_CUBE, rho, dx, dy, dz,
                  xlim, ylim, zlim,
                  vmin=None, vmax=None,
                  opacity_func=None,
                  colorscale='hot',
                  showaxis=True, showColorbar=True,
                  width_screen=1000, height_screen=800,
                  backgroundcolor='white', fontcolor='black',
                  colorbar_title='λ (mrad²/cm)',
                  ceye=None, ccenter=None, cup=None,
                  cbar_location=None,
                  png_filename=None, png_scale=5,
                  sentinel=-1.0,
                  abnormalRegion_vertices=None,
                  abnormalRegion_width=4,
                  abnormalRegion_color=None,
                  abnormalRegion_names=None,
                  legend_location=None):
    """
    3D voxel rendering of ML-EM scattering density reconstruction.

    Parameters
    ----------
    abnormalRegion_vertices : list of np.ndarray, optional
        List of 8-vertex arrays defining cuboids to highlight with wireframe.
        Use get_cuboid_vertices() to generate from (center, size) pairs.
    abnormalRegion_width : float or list of float
        Edge width(s). Single value broadcasts to all regions.
    abnormalRegion_color : str or list of str, optional
        Edge color(s). Default is 'black' for all.
    abnormalRegion_names : list of str, optional
        Legend names for highlighted regions.
    legend_location : [x, y], optional
        Legend position in figure coordinates.
    colorbar_title : str
        Plotly does NOT support LaTeX. Use Unicode: 'λ (mrad²/cm)'.
    """
    normlim = np.sqrt((xlim[1]-xlim[0])**2 + (ylim[1]-ylim[0])**2 + (zlim[1]-zlim[0])**2)
    x_range = (xlim[1]-xlim[0]) / normlim
    y_range = (ylim[1]-ylim[0]) / normlim
    z_range = (zlim[1]-zlim[0]) / normlim

    Xf = X_CUBE.ravel()
    Yf = Y_CUBE.ravel()
    Zf = Z_CUBE.ravel()
    rho_f = rho.ravel()

    seen_flat = rho_f > sentinel + 0.5

    if vmin is None:
        vmin = rho_f[seen_flat].min() if np.any(seen_flat) else 0.0
    if vmax is None:
        vmax = rho_f[seen_flat].max() if np.any(seen_flat) else 1.0

    print(f"Value range (seen voxels): [{vmin:.4f}, {vmax:.4f}]")
    print(f"Seen voxels: {np.sum(seen_flat)}, Skipped (sentinel): {np.sum(~seen_flat)}")

    rho_norm = np.zeros_like(rho_f)
    if vmax > vmin:
        rho_norm[seen_flat] = (rho_f[seen_flat] - vmin) / (vmax - vmin)
    rho_norm = np.clip(rho_norm, 0.0, 1.0)

    if opacity_func is None:
        opacity_func = lambda rho_val, alpha, idx: min(alpha * 0.8 + 0.1, 0.95)

    title_fontsize = 18
    tick_fontsize = 16

    layout = go.Layout(
        scene=dict(
            xaxis=dict(
                title=dict(text='X (cm)', font=dict(size=title_fontsize, family='Arial_Bold')) if showaxis else dict(text=''),
                tickfont=dict(size=tick_fontsize, family='Arial'),
                showline=showaxis, linecolor='black', linewidth=4,
                showticklabels=showaxis, zeroline=False, showspikes=False,
                ticks='outside',
                tickcolor='black' if showaxis else backgroundcolor,
                showgrid=False, showbackground=False,
                range=[xlim[0], xlim[1]]
            ),
            yaxis=dict(
                title=dict(text='Y (cm)', font=dict(size=title_fontsize, family='Arial_Bold')) if showaxis else dict(text=''),
                tickfont=dict(size=tick_fontsize, family='Arial'),
                showline=showaxis, linecolor='black', linewidth=4,
                showticklabels=showaxis, zeroline=False, showspikes=False,
                ticks='outside',
                tickcolor='black' if showaxis else backgroundcolor,
                showgrid=False, showbackground=False,
                range=[ylim[0], ylim[1]]
            ),
            zaxis=dict(
                title=dict(text='Z (cm)', font=dict(size=title_fontsize, family='Arial_Bold')) if showaxis else dict(text=''),
                tickfont=dict(size=tick_fontsize, family='Arial'),
                showline=showaxis, linecolor='black', linewidth=4,
                showticklabels=showaxis, zeroline=False, showspikes=False,
                ticks='outside',
                tickcolor='black' if showaxis else backgroundcolor,
                showgrid=False, showbackground=False,
                range=[zlim[0], zlim[1]]
            ),
            aspectmode='manual',
            aspectratio=dict(x=x_range, y=y_range, z=z_range),
            camera=dict(
                eye=ceye if ceye is not None else dict(x=0.5, y=0.45, z=0.3),
                up=dict(x=0, y=0, z=1) if cup is None else cup,
                center=ccenter if ccenter is not None else dict(x=0, y=0, z=-0.01)
            )
        ),
        paper_bgcolor=backgroundcolor,
        legend=dict(
            x=0.7 if legend_location is None else legend_location[0],
            y=1.0 if legend_location is None else legend_location[1],
            font=dict(size=20, color=fontcolor, family='Arial Black'),
            itemwidth=30,
        )
    )

    # Build voxel meshes
    cubes = []
    for i in range(len(rho_f)):
        if not seen_flat[i]:
            continue
        alpha_val = rho_norm[i]
        opacity_val = opacity_func(rho_f[i], alpha_val, i)
        if opacity_val < 0.01:
            continue

        x_ctr, y_ctr, z_ctr = Xf[i], Yf[i], Zf[i]
        x0, x1 = x_ctr - dx/2, x_ctr + dx/2
        y0, y1 = y_ctr - dy/2, y_ctr + dy/2
        z0, z1 = z_ctr - dz/2, z_ctr + dz/2

        points = np.array([
            [x0, y0, z0], [x1, y0, z0], [x1, y1, z0], [x0, y1, z0],
            [x0, y0, z1], [x1, y0, z1], [x1, y1, z1], [x0, y1, z1]
        ])

        cubes.append(go.Mesh3d(
            x=points[:, 0], y=points[:, 1], z=points[:, 2],
            i=[0, 0, 4, 4, 0, 0, 3, 3, 0, 0, 1, 1],
            j=[1, 2, 5, 6, 1, 5, 2, 6, 3, 7, 2, 6],
            k=[2, 3, 6, 7, 5, 4, 6, 7, 7, 4, 6, 5],
            intensity=[alpha_val] * 8,
            opacity=opacity_val,
            colorscale=colorscale,
            showscale=False,
            cmin=0.0, cmax=1.0,
        ))

    # Dummy trace for colorbar
    colorbar_trace = go.Mesh3d(
        x=[0, 1, 2, 0], y=[0, 0, 1, 2], z=[0, 2, 0, 1],
        i=[0, 0, 0, 1], j=[1, 2, 3, 2], k=[2, 3, 1, 3],
        opacity=0,
        colorscale=colorscale,
        intensity=[vmin,
                   vmin + 0.2 * (vmax - vmin),
                   vmin + 0.4 * (vmax - vmin),
                   vmin + 0.6 * (vmax - vmin),
                   vmin + 0.8 * (vmax - vmin),
                   vmax],
        colorbar=dict(
            title=dict(text=colorbar_title,
                       font=dict(size=20, color=fontcolor, family='Arial Black')),
            tickfont=dict(size=20, color=fontcolor),
            thickness=20, len=0.5,
            x=0.85 if cbar_location is None else cbar_location[0],
            y=0.5 if cbar_location is None else cbar_location[1],
            yanchor='middle'
        ),
        showscale=showColorbar,
    )

    fig_data = [colorbar_trace] + cubes
    fig = go.Figure(data=fig_data, layout=layout)

    # ============================================================
    # Abnormal-region wireframe overlays
    # ============================================================
    if abnormalRegion_vertices is not None:
        n = len(abnormalRegion_vertices)
        legend_name = abnormalRegion_names if abnormalRegion_names is not None else None

        if isinstance(abnormalRegion_width, (list, tuple, np.ndarray)):
            if len(abnormalRegion_width) != n:
                raise ValueError(f"Width count ({len(abnormalRegion_width)}) and region count ({n}) mismatch")
            widths = abnormalRegion_width
        else:
            widths = [abnormalRegion_width] * n

        if abnormalRegion_color is None:
            colors = ['black'] * n
        elif isinstance(abnormalRegion_color, (list, tuple, np.ndarray)):
            if len(abnormalRegion_color) != n:
                raise ValueError(f"Color count ({len(abnormalRegion_color)}) and region count ({n}) mismatch")
            colors = abnormalRegion_color
        else:
            colors = [abnormalRegion_color] * n

        legend_size = 20
        for i, (vertex, width, color) in enumerate(zip(abnormalRegion_vertices, widths, colors)):
            fig.add_traces(plot_cube_edges(vertex, color=color, width=width))
            fig.add_trace(plot_cube_edges_legend(
                size=legend_size, color=color,
                name=legend_name[i] if legend_name is not None else f'Region {i+1}'
            ))

    if png_filename is not None:
        pio.write_image(fig, png_filename,
                        width=width_screen, height=height_screen, scale=png_scale)
        print(f"Saved to: {png_filename}")
    else:
        fig.update_layout(width=width_screen, height=height_screen)
        fig.show()

    return fig

def get_cuboid_vertices(positions, sizes):
    """
    Compute 8 vertex coordinates of a cube
    Vertex order: 4 bottom face points (CCW) + 4 top face points (CCW)
    """
    vertices = []
    for pos, size in zip(positions, sizes):
        x, y, z = pos
        l, w, h = size
        
        # Half side lengths
        half_l, half_w, half_h = l/2, w/2, h/2
        
        # 8 vertices: 4 bottom face + 4 top face
        verts = np.array([
            [x, y, z],  # Bottom-left-rear
            [x + 2*half_l, y, z],  # Bottom-right-rear
            [x + 2*half_l, y + 2*half_w, z],  # Right-rear
            [x, y + 2*half_w, z],  # Left-rear
            [x, y, z + 2*half_h],  # Bottom-left-front
            [x + 2*half_l, y, z + 2*half_h],  # Bottom-right-front
            [x + 2*half_l, y + 2*half_w, z + 2*half_h],  # Right-front
            [x, y + 2*half_w, z + 2*half_h]   # Left-front
        ])
        vertices.append(verts)
    
    return vertices
# @numba.njit
def vertices_to_aabb(vertices):
    mins = vertices.min(axis=0)
    maxs = vertices.max(axis=0)
    return mins, maxs

@numba.njit
def point_in_any_abnormal_region(point, abnormal_aabbs):
    x, y, z = point
    for (mins, maxs) in abnormal_aabbs:
        if (
            mins[0] <= x <= maxs[0] and
            mins[1] <= y <= maxs[1] and
            mins[2] <= z <= maxs[2]
        ):
            return True
    return False

@numba.njit
def voxel_in_abnormal_region(center, dx, dy, dz, abnormal_mins, abnormal_maxs, threshold):
    x, y, z = center
    hx = dx * 0.5
    hy = dy * 0.5
    hz = dz * 0.5

    count = 0

    for ix in (-1, 0, 1):
        for iy in (-1, 0, 1):
            for iz in (-1, 0, 1):
                px = x + ix * hx
                py = y + iy * hy
                pz = z + iz * hz

                for k in range(abnormal_mins.shape[0]):
                    if (
                        abnormal_mins[k, 0] <= px <= abnormal_maxs[k, 0] and
                        abnormal_mins[k, 1] <= py <= abnormal_maxs[k, 1] and
                        abnormal_mins[k, 2] <= pz <= abnormal_maxs[k, 2]
                    ):
                        count += 1
                        break

                if count >= threshold:
                    return True

    return False


def collect_voxel_labels(
    X_CUBE, Y_CUBE, Z_CUBE,
    rho,
    dx, dy, dz,
    abnormalRegion_vertices,
    vertex_threshold=5,
    ignore=-1
):
    Xf = X_CUBE.ravel()
    Yf = Y_CUBE.ravel()
    Zf = Z_CUBE.ravel()
    rho_f = rho.ravel()

    # Pre-compute AABB
    abnormal_mins = []
    abnormal_maxs = []
    for v in abnormalRegion_vertices:
        mins, maxs = vertices_to_aabb(v)
        abnormal_mins.append(mins)
        abnormal_maxs.append(maxs)

    abnormal_mins = np.array(abnormal_mins)
    abnormal_maxs = np.array(abnormal_maxs)

    rho_list = []
    gt_list = []

    for i in range(len(rho_f)):
        if rho_f[i] == ignore:
            continue

        center = (Xf[i], Yf[i], Zf[i])
        in_abnormal = voxel_in_abnormal_region(
            center, dx, dy, dz,
            abnormal_mins, abnormal_maxs,
            vertex_threshold
        )

        rho_list.append(rho_f[i])
        gt_list.append(in_abnormal)

    return np.array(rho_list), np.array(gt_list)

def compute_roc_curve(
    rho_vals, gt,
    thresholds,
    mode = "greater"
):
    order = np.argsort(rho_vals)
    rho_sorted = rho_vals[order]
    gt_sorted = gt[order]

    if mode == "greater":
        tp_cum = np.cumsum(gt_sorted[::-1])
        fp_cum = np.cumsum((~gt_sorted)[::-1])
    elif mode == "less":
        tp_cum = np.cumsum(gt_sorted)
        fp_cum = np.cumsum(~gt_sorted)
    else:
        raise ValueError("mode must be 'greater' or 'less'")

    total_pos = gt_sorted.sum()
    total_neg = len(gt_sorted) - total_pos

    fprs = []
    tprs = []
    prrs = []

    if mode == "greater":
        for t in thresholds:
            # Find count of rho > t
            k = np.searchsorted(rho_sorted, t, side="right", sorter=None)
            # Since it's descending, reverse the position
            k = len(rho_sorted) - k

            if k <= 0:
                fprs.append(0.0)
                tprs.append(0.0)
                prrs.append(0.0)
            else:
                fprs.append(fp_cum[k-1] / total_neg)
                tprs.append(tp_cum[k-1] / total_pos)
                prrs.append(tp_cum[k-1] / (tp_cum[k-1] + fp_cum[k-1]))
                print(f"Threshold: {t:.3f}, TP: {tp_cum[k-1]}, FP: {fp_cum[k-1]}, FPR: {fprs[-1]:.3f}, TPR: {tprs[-1]:.3f}, PRR: {prrs[-1]:.3f}")
    elif mode == "less":
        for t in thresholds:
            # Find count of rho < t
            k = np.searchsorted(rho_sorted, t, side="left")
            
            if k <= 0:
                fprs.append(0.0)
                tprs.append(0.0)
                prrs.append(0.0)
            else:
                fprs.append(fp_cum[k-1] / total_neg)
                tprs.append(tp_cum[k-1] / total_pos)
                prrs.append(tp_cum[k-1] / (tp_cum[k-1] + fp_cum[k-1]))
                print(f"Threshold: {t:.3f}, TP: {tp_cum[k-1]}, FP: {fp_cum[k-1]}, FPR: {fprs[-1]:.3f}, TPR: {tprs[-1]:.3f}, PRR: {prrs[-1]:.3f}")

    return {
        "fpr": np.array(fprs),
        "tpr": np.array(tprs),
        "prr": np.array(prrs),
        "thresholds": np.array(thresholds)
    }

In [44]:
# ============================================================
# 3D Reconstruction Visualization
# ============================================================

print("\n--- 3D Reconstruction (Print_result Style) ---")

# Set visualization bounds
xlim = [X_MIN, X_MAX]
ylim = [Y_MIN, Y_MAX]
zlim = [Z_MIN, Z_MAX]

def opacity_fn(rho_val, alpha, idx):
    # Hide sentinel (unseen) and low-scatter voxels
    if rho_val < 0:
        return 0.0
    elif rho_val < 3:
        return 0.0
    return alpha

# Camera settings for a good initial viewpoint
view_zoom = 1.0
ceye = dict(x=0.5 * view_zoom, y=0.45 * view_zoom, z=0.3 * view_zoom)
cup = dict(x=0, y=0, z=1)

# ---- 可选: 添加异常区域框线 ----
# 使用方法 (与 Example2_visualize.ipynb 完全一致):
#
#   sample_positions = [(x1, y1, z1), (x2, y2, z2), ...]   # 立方体中心坐标
#   sample_sizes      = [(lx1, ly1, lz1), (lx2, ly2, lz2), ...]  # 立方体尺寸
#   vertices          = get_cuboid_vertices(sample_positions, sample_sizes)
#
#   然后在 PrintMLResult 调用中添加:
#     abnormalRegion_vertices=vertices,
#     abnormalRegion_width=[5, 5, ...],          # 线宽
#     abnormalRegion_color=['red', 'blue', ...],  # 颜色
#     abnormalRegion_names=['Box A', 'Box B', ...],  # 图例名

# 示例: 画一个 10×10×10 cm 的框
sample_positions_example = [(0.1 ,0.1 , -0.1),
    (-0.1 ,-0.3 , -0.1),]
sample_sizes_example = [(0.1, 0.1, 0.1),
    (0.2, 0.2, 0.2),]
sample_positions_example = np.array(sample_positions_example) * 100
sample_positions_example = [tuple(row) for row in sample_positions_example]
sample_sizes_example = np.array(sample_sizes_example) * 100
sample_sizes_example = [tuple(row) for row in sample_sizes_example]
example_vertices = get_cuboid_vertices(sample_positions_example, sample_sizes_example)

fig_3d = PrintMLResult(
    X_CUBE, Y_CUBE, Z_CUBE, rho,
    dx, dy, dz,
    xlim, ylim, zlim,
    vmin=0, vmax=rho.max(),
    opacity_func=opacity_fn,
    colorscale='rainbow',
    width_screen=1000, height_screen=800,
    showaxis=True, showColorbar=True,
    ceye=ceye, cup=cup,
    backgroundcolor='white', fontcolor='black',
    sentinel=Sentinel,
    abnormalRegion_vertices=example_vertices,
    abnormalRegion_width=7,
    abnormalRegion_color=['red', 'blue'],
    abnormalRegion_names=['Target region1', 'Target region2'],
)

print("\n=== 3D Visualization Complete ===")


--- 3D Reconstruction (Print_result Style) ---
Value range (seen voxels): [0.0000, 20.9644]
Seen voxels: 1000, Skipped (sentinel): 0



=== 3D Visualization Complete ===


In [33]:
# Compute reconstruct precision
sample_positions_cube = [
    (0.1 ,0.1 , -0.1),
    (-0.1 ,-0.3 , -0.1),
]
sample_sizes_cube = [
    (0.1, 0.1, 0.1),
    (0.2, 0.2, 0.2),
]
arr = np.array(sample_positions_cube) * 100
sample_positions_cube = [tuple(row) for row in arr]
arr = np.array(sample_sizes_cube) * 100
sample_sizes_cube = [tuple(row) for row in arr]

vertices = get_cuboid_vertices(sample_positions_cube, sample_sizes_cube)

valid_rho = rho[seen_mask]
valid_rho_max = valid_rho.max()
valid_rho_min = valid_rho.min()
print("Valid rho max:", valid_rho_max)
print("Valid rho min:", valid_rho_min)
# thresholds = np.array([1.5,2])
thresholds = np.array([1, 10])

rho_vals, gt = collect_voxel_labels(
    X_CUBE, Y_CUBE, Z_CUBE,
    rho,
    dx, dy, dz,
    abnormalRegion_vertices=vertices,
    vertex_threshold=6,
    ignore=Sentinel
)

roc_data = compute_roc_curve(
    rho_vals, gt,
    thresholds,
    mode="greater"
    # mode="less"
)
print("pr", roc_data["prr"])
print("tpr", roc_data["tpr"])

Valid rho max: 20.964383777934344
Valid rho min: 0.10345613273164024
Threshold: 1.000, TP: 62, FP: 64, FPR: 0.069, TPR: 0.873, PRR: 0.492
Threshold: 10.000, TP: 7, FP: 0, FPR: 0.000, TPR: 0.099, PRR: 1.000
pr [0.49206349 1.        ]
tpr [0.87323944 0.09859155]


In [ ]:
# ============================================================
# 2D Slice Visualization
# ============================================================

def plot_2d_slices(rho, X_CUBE, Y_CUBE, Z_CUBE,
                   slice_axis='z', num_slices=6, slice_positions=None,
                   figsize=None, cmap='viridis', log_scale=False,
                   vmin=None, vmax=None, title_prefix="",
                   outputfilepath=None, sentinel=-1.0):
    """
    Draw 2D slices of 3D scattering density data.

    Voxels with value == sentinel (default -1, meaning "no ray data")
    are rendered as NaN (transparent/masked) in slices.
    """
    plt.ioff()

    if slice_axis.lower() == 'x':
        x_label, y_label = 'Y (cm)', 'Z (cm)'
        x_coord, y_coord = Y_CUBE, Z_CUBE
        axis_name = 'X (cm)'
        axis_coords = X_CUBE[:, 0, 0]
    elif slice_axis.lower() == 'y':
        x_label, y_label = 'X (cm)', 'Z (cm)'
        x_coord, y_coord = X_CUBE, Z_CUBE
        axis_name = 'Y (cm)'
        axis_coords = Y_CUBE[0, :, 0]
    elif slice_axis.lower() == 'z':
        x_label, y_label = 'X (cm)', 'Y (cm)'
        x_coord, y_coord = X_CUBE, Y_CUBE
        axis_name = 'Z (cm)'
        axis_coords = Z_CUBE[0, 0, :]
    else:
        raise ValueError("slice_axis must be 'x', 'y', or 'z'")

    # Determine slice positions
    if slice_positions is None:
        indices = np.linspace(0, len(axis_coords)-1, num_slices, dtype=int)
        slice_positions = axis_coords[indices]
        slice_indices = indices
    else:
        slice_indices = []
        actual_positions = []
        for pos in slice_positions:
            idx = np.argmin(np.abs(axis_coords - pos))
            slice_indices.append(idx)
            actual_positions.append(axis_coords[idx])
        slice_positions = actual_positions

    # Deduplicate
    slice_indices, unique_idx = np.unique(slice_indices, return_index=True)
    slice_positions = [slice_positions[i] for i in unique_idx]

    n_slices = len(slice_positions)
    cols = int(np.ceil(np.sqrt(n_slices)))
    rows = int(np.ceil(n_slices / cols))

    if figsize is None:
        figsize = (4*cols, 4*rows)

    fig, axes = plt.subplots(rows, cols, figsize=figsize)
    if n_slices == 1:
        axes = [axes]
    else:
        axes = axes.flatten()

    if vmin is None:
        vmin = np.maximum(np.min(rho[rho >= 0]), 0) if np.any(rho >= 0) else 0
    if vmax is None:
        vmax = np.max(rho[rho >= 0]) if np.any(rho >= 0) else 1

    for i, (idx, pos) in enumerate(zip(slice_indices, slice_positions)):
        if i >= len(axes):
            break
        ax = axes[i]

        # Extract slice
        if slice_axis.lower() == 'x':
            slice_data = rho[idx, :, :].copy()
            x_slice = x_coord[idx, :, :]
            y_slice = y_coord[idx, :, :]
        elif slice_axis.lower() == 'y':
            slice_data = rho[:, idx, :].copy()
            x_slice = x_coord[:, idx, :]
            y_slice = y_coord[:, idx, :]
        else:
            slice_data = rho[:, :, idx].copy()
            x_slice = x_coord[:, :, idx]
            y_slice = y_coord[:, :, idx]

        # Filter sentinel (unseen voxels) to NaN for transparent display
        slice_data[slice_data <= sentinel + 0.5] = np.nan

        # Mask near-zero values for cleaner display
        slice_data_disp = np.where(slice_data > vmin * 0.01, slice_data, np.nan)

        if log_scale and np.any(slice_data_disp > 0):
            norm = LogNorm(vmin=max(vmin, 1e-6), vmax=vmax)
            im = ax.pcolormesh(x_slice, y_slice, slice_data_disp,
                              cmap=cmap, norm=norm, shading='auto')
        else:
            im = ax.pcolormesh(x_slice, y_slice, slice_data_disp,
                              cmap=cmap, vmin=vmin, vmax=vmax, shading='auto')

        ax.set_title(f'{title_prefix}{axis_name} = {pos:.1f}')
        ax.set_xlabel(x_label)
        ax.set_ylabel(y_label)
        ax.set_aspect('equal')

        print(f"Slice: {title_prefix}{axis_name} = {pos:.2f}, index: {idx}")

    # Colorbar
    cbar = fig.colorbar(im, ax=axes[:n_slices].tolist(), shrink=0.8, aspect=20)
    cbar.set_label(r'$\lambda$ (mrad$^2$/cm)')

    # Hide extra subplots
    for i in range(n_slices, len(axes)):
        axes[i].set_visible(False)

    if outputfilepath is not None:
        fig.savefig(outputfilepath, bbox_inches='tight', dpi=300)
        print(f"Saved to: {outputfilepath}")

    plt.tight_layout()
    return fig, axes

In [ ]:
# ============================================================
# Quick Diagnostic Plots
# ============================================================

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# 1. Histogram of λ values (seen voxels only, excludes -1 sentinel)
ax = axes[0, 0]
seen_vals = rho[seen_mask]
ax.hist(seen_vals, bins=60, edgecolor='black', alpha=0.7)
ax.axvline(Lambda_init, color='red', linestyle='--', label=f'λ_init={Lambda_init}')
ax.set_xlabel(r'Scattering density $\lambda$ (mrad$^2$/cm)')
ax.set_ylabel('Voxel count')
n_unseen_hist = np.sum(~seen_mask)
ax.set_title(f'Histogram of reconstructed λ\n({n_unseen_hist} unseen voxels excluded)')
ax.legend()
ax.set_yscale('log')

# 2. Z-slice at midpoint — unseen voxels shown as gray
ax = axes[0, 1]
k_mid = N_Z // 2
slice_data = rho[:, :, k_mid]
slice_masked = np.ma.masked_where(~seen_mask[:, :, k_mid], slice_data)
im = ax.pcolormesh(
    np.linspace(X_MIN, X_MAX, N_X + 1),
    np.linspace(Y_MIN, Y_MAX, N_Y + 1),
    slice_masked.T,
    cmap='hot', shading='auto')
# Overlay gray for unseen voxels
slice_unseen = ~seen_mask[:, :, k_mid]
if np.any(slice_unseen):
    ax.pcolormesh(
        np.linspace(X_MIN, X_MAX, N_X + 1),
        np.linspace(Y_MIN, Y_MAX, N_Y + 1),
        np.where(slice_unseen, 1.0, np.nan).T,
        cmap='Greys', shading='auto', alpha=0.3, vmin=0, vmax=1)
ax.set_xlabel('X (cm)')
ax.set_ylabel('Y (cm)')
ax.set_title(f'λ at z ≈ {Z_MIN + (k_mid + 0.5)*dz:.1f} cm\n(gray = unseen)')
ax.set_aspect('equal')
plt.colorbar(im, ax=ax, label=r'$\lambda$ (mrad$^2$/cm)')

# 3. Y-slice at midpoint — unseen voxels shown as gray
ax = axes[1, 0]
j_mid = N_Y // 2
slice_data_y = rho[:, j_mid, :]
slice_masked_y = np.ma.masked_where(~seen_mask[:, j_mid, :], slice_data_y)
im2 = ax.pcolormesh(
    np.linspace(X_MIN, X_MAX, N_X + 1),
    np.linspace(Z_MIN, Z_MAX, N_Z + 1),
    slice_masked_y.T,
    cmap='hot', shading='auto')
slice_unseen_y = ~seen_mask[:, j_mid, :]
if np.any(slice_unseen_y):
    ax.pcolormesh(
        np.linspace(X_MIN, X_MAX, N_X + 1),
        np.linspace(Z_MIN, Z_MAX, N_Z + 1),
        np.where(slice_unseen_y, 1.0, np.nan).T,
        cmap='Greys', shading='auto', alpha=0.3, vmin=0, vmax=1)
ax.set_xlabel('X (cm)')
ax.set_ylabel('Z (cm)')
ax.set_title(f'λ at y ≈ {Y_MIN + (j_mid + 0.5)*dy:.1f} cm\n(gray = unseen)')
ax.set_aspect('equal')
plt.colorbar(im2, ax=ax, label=r'$\lambda$ (mrad$^2$/cm)')

# 4. Depth profile (mean λ per Z-slice, seen voxels only)
ax = axes[1, 1]
z_centers = np.linspace(Z_MIN + dz/2, Z_MAX - dz/2, N_Z)
mean_per_z = np.array([
    rho[:, :, k][seen_mask[:, :, k]].mean() if np.any(seen_mask[:, :, k]) else np.nan
    for k in range(N_Z)])
max_per_z = np.array([
    rho[:, :, k][seen_mask[:, :, k]].max() if np.any(seen_mask[:, :, k]) else np.nan
    for k in range(N_Z)])
ax.plot(z_centers, mean_per_z, 'o-', label='Mean λ', markersize=3)
ax.plot(z_centers, max_per_z, 's-', label='Max λ', markersize=3)
ax.set_xlabel('Z (cm)')
ax.set_ylabel(r'$\lambda$ (mrad$^2$/cm)')
ax.set_title('Depth profile of scattering density (seen only)')
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# Multi-slice Views
# ============================================================

print("\n--- XY Slices (Z-axis) ---")
fig_xy, _ = plot_2d_slices(rho, X_CUBE, Y_CUBE, Z_CUBE,
                           slice_axis='z', num_slices=6,
                           cmap='hot', title_prefix="Z=")
plt.show()

print("\n--- XZ Slices (Y-axis) ---")
fig_xz, _ = plot_2d_slices(rho, X_CUBE, Y_CUBE, Z_CUBE,
                           slice_axis='y', num_slices=5,
                           cmap='hot', title_prefix="Y=")
plt.show()